
# Walmart Global Model Sanity Check

이 notebook의 목적은 하나입니다.

`patchtst_exo`, `timexer`, `exotst` 같은 **global model family 자체가 구조적으로 문제인지**, 아니면
현재 DSIO 부품 데이터의 `small / erratic / high-obsolescence / decline` 세그먼트가 문제인지 분리해서 보는 것입니다.

검증 논리는 단순합니다.
- Walmart처럼 비교적 dense하고 규칙적인 weekly retail 데이터에서도
  글로벌 모델이 비슷하게 심한 overforecast / underforecast를 만들면 모델/학습 파이프라인 문제일 가능성이 큽니다.
- 반대로 Walmart에서는 aggregate scale과 part-level shape가 비교적 정상적이면,
  현재 DSIO 문제는 모델 family 자체보다 **데이터 세그먼트 특성** 쪽일 가능성이 높습니다.

권장 실행 순서:
1. Setup / Config
2. Data Preparation
3. Loader Smoke Test
4. Train + Forecast
5. Metrics / Plots
6. Interpretation Guide


In [ ]:

from __future__ import annotations

%load_ext autoreload
%autoreload 2

import argparse
import importlib.util
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
from IPython.display import Image, display

REPO_ROOT_OVERRIDE = None


def _looks_like_repo(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src').exists()


def _iter_named_repo_candidates(root: Path, repo_name: str = 'ts_forecaster_lib', max_depth: int = 4):
    if not root.exists() or not root.is_dir():
        return
    try:
        root_resolved = root.resolve()
    except Exception:
        root_resolved = root
    stack = [(root_resolved, 0)]
    while stack:
        current, depth = stack.pop()
        if current.name == repo_name:
            yield current
        if depth >= max_depth:
            continue
        try:
            children = list(current.iterdir())
        except Exception:
            continue
        for child in children:
            if child.is_dir() and not child.name.startswith('.'):
                stack.append((child, depth + 1))


def find_repo_root(start: Path, explicit_repo_root: Path | None = None) -> Path | None:
    env_repo_root = os.environ.get('TS_FORECASTER_REPO_ROOT')
    candidates = []
    if explicit_repo_root is not None:
        candidates.append(Path(explicit_repo_root).expanduser().resolve())
    if env_repo_root:
        candidates.append(Path(env_repo_root).expanduser().resolve())
    candidates.extend([start, *start.parents])

    home = Path.home()
    common_roots = [
        home,
        home / 'workspace',
        home / 'workspaces',
        home / 'projects',
        home / 'PycharmProjects',
        Path('/workspace'),
        Path('/workspaces'),
        Path('/home'),
        Path('/root'),
    ]
    for root in common_roots:
        candidates.extend(_iter_named_repo_candidates(root))

    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if _looks_like_repo(candidate):
            return candidate
    return None


def resolve_import_paths() -> tuple[Path | None, Path | None]:
    repo_root = find_repo_root(Path.cwd().resolve(), explicit_repo_root=REPO_ROOT_OVERRIDE)
    src_root = repo_root / 'src' if repo_root is not None else None
    if src_root is not None and str(src_root) not in sys.path:
        sys.path.insert(0, str(src_root))

    spec = importlib.util.find_spec('modeling_module')
    if spec is None or spec.origin is None:
        raise RuntimeError(
            'Could not import modeling_module. Set REPO_ROOT_OVERRIDE, TS_FORECASTER_REPO_ROOT, or install the package.'
        )

    module_init = Path(spec.origin).resolve()
    module_root = module_init.parent
    if repo_root is None and module_root.parent.name == 'src':
        repo_root = module_root.parent.parent
        src_root = repo_root / 'src'
    return repo_root, src_root


NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT, SRC_ROOT = resolve_import_paths()

from modeling_module import (
    ArtifactConfig,
    DataColumnConfig,
    DataRequest,
    DataWindowConfig,
    ExogenousConfig,
    LoaderConfig,
    RuntimeConfig,
    SSLConfig,
    TrainRequest,
    TrainerConfig,
)
from modeling_module.api.data import build_datamodule
from modeling_module.api.infer import load_predictor
from modeling_module.api.train import train
from model_test.exogenous_test.exogenous_ab_utils import (
    attach_actuals,
    compute_metric_tables,
    ensure_dir,
    make_point_forecast_result_table,
    plot_latest_revision_aggregate,
    plot_latest_revision_part_grid,
    plot_metric_summary,
    resolve_single_plan_week,
    select_eval_ids_with_full_actual_coverage,
    select_latest_revision,
    select_plot_parts,
)
from model_test.exogenous_test.run_exogenous_model_ab import (
    ModelSpec,
    build_architecture_config,
)

WALMART_MODEL_SPECS: dict[str, ModelSpec] = {
    'patchtst_no_future': ModelSpec(label='patchtst_no_future', request_key='patchtst_base', use_future_exogenous=False),
    'patchtst_head_flatten': ModelSpec(label='patchtst_head_flatten', request_key='patchtst_base', use_future_exogenous=True),
    'patchtst_token_cross_attn': ModelSpec(label='patchtst_token_cross_attn', request_key='patchtst_base', use_future_exogenous=True),
    'timexer': ModelSpec(label='timexer', request_key='timexer_base', use_future_exogenous=False),
    'exotst': ModelSpec(label='exotst', request_key='exotst_base', use_future_exogenous=True),
}

WALMART_PATCHTST_FUSION_MODES: dict[str, str | None] = {
    'patchtst_no_future': None,
    'patchtst_head_flatten': 'head_flatten',
    'patchtst_token_cross_attn': 'token_cross_attn',
}


def resolve_walmart_model_specs(names: list[str]) -> list[ModelSpec]:
    out: list[ModelSpec] = []
    for name in names:
        key = str(name).strip().lower()
        if key not in WALMART_MODEL_SPECS:
            raise ValueError(f'Unsupported Walmart model name: {name}. Expected one of {sorted(WALMART_MODEL_SPECS)}')
        out.append(WALMART_MODEL_SPECS[key])
    return out


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def configure_torch_runtime() -> tuple[str, str | None]:
    if torch.cuda.is_available():
        torch.set_float32_matmul_precision('high')
        return 'cuda', None
    if torch.backends.mps.is_available():
        return 'mps', 'CUDA not available, using Apple MPS.'
    return 'cpu', 'CUDA/MPS not available, using CPU.'



## Config

이 notebook은 Walmart weekly dense dataset에서 **global model sanity check**를 수행합니다.

기본 해석 기준:
- `pred_to_actual_ratio`가 1 근처면 total scale이 비교적 정상
- aggregate plot이 GT shape를 비슷하게 따라가면 global model family 자체 문제 가능성은 낮음
- 여기서도 DSIO와 비슷하게 강한 overforecast가 반복되면 모델/학습 파이프라인을 더 의심해야 함


In [ ]:

artifact_root = REPO_ROOT / 'artifacts' / 'exogenous_test' / 'walmart_global_sanity'
ensure_dir(artifact_root)

PARQUET_PATH = REPO_ROOT / 'raw_data' / 'train_data' / 'walmart_best_feature_train.parquet'

models_to_run = [
    'patchtst_no_future',
    # 'patchtst_head_flatten',
    'patchtst_token_cross_attn',
    'timexer',
    'exotst'
]

lookback = 52
horizon = 13
plan_week = 201231
plot_part_count = 12
max_parts_per_plan = 10_000

train_batch_size = 128
infer_batch_size = 64
num_workers = 0
pin_memory = False
persistent_workers = False
prefetch_factor = 2
seed = 22

warmup_epochs = 30
spike_epochs = 5
lr = 1e-3
patchtst_future_exo_fusion_dropout = 0.1
ssl_mode = 'full'
ssl_pretrain_epochs = 0
ssl_mask_ratio = 0.3
ssl_loss_type = 'mse'

# Walmart sanity check에서는 dense/continuous data에 맞춰 보수적인 설정으로 시작합니다.
use_intermittent = False
val_use_weights = False

set_global_seed(seed)
default_device, device_note = configure_torch_runtime()
device = default_device
effective_patchtst_ssl_mode = ssl_mode if (str(ssl_mode).lower() != 'sl_only' and int(ssl_pretrain_epochs) > 0) else 'sl_only'

print('PARQUET_PATH      :', PARQUET_PATH)
print('models_to_run     :', models_to_run)
print('lookback / horizon:', lookback, horizon)
print('plan_week         :', plan_week)
print('device            :', device)
print('ssl_mode(requested):', ssl_mode)
print('ssl_mode(effective):', effective_patchtst_ssl_mode)
print('ssl_pretrain_epochs:', ssl_pretrain_epochs)
print('use_intermittent  :', use_intermittent)
print('val_use_weights   :', val_use_weights)
if device_note:
    print('device_note       :', device_note)



## Data Preparation

Walmart parquet은 이미 best feature train table 형태이므로,
이 notebook에서는 별도의 lifecycle 가공 없이 바로 global model 검증에 사용합니다.

컬럼 사용 규칙:
- `y`: target
- `exo_p_*`: past exogenous
- `exo_*` but not `exo_p_*`, `exo_c_*`: future-known continuous exogenous
- `exo_c_*`: 원래는 categorical 후보지만, 이 sanity check에서는 cardinality/indexer 의존성을 줄이기 위해 **continuous로 흡수**합니다.


In [ ]:

ID_COL = 'unique_id'
DATE_COL = 'date'
Y_COL = 'y'
FREQ = 'weekly'

df = (
    pl.read_parquet(PARQUET_PATH)
    .with_columns([
        pl.col(ID_COL).cast(pl.String),
        pl.col(DATE_COL).cast(pl.Int64),
        pl.col(Y_COL).cast(pl.Float64),
        *[pl.col(c).cast(pl.Float64) for c in pl.read_parquet(PARQUET_PATH, n_rows=1).columns if c.startswith('exo_c_')],
    ])
    .sort([ID_COL, DATE_COL])
)

target_df = df.select([ID_COL, DATE_COL, Y_COL])

past_exo_cont_cols = [c for c in df.columns if c.startswith('exo_p_')] + [c for c in df.columns if c.startswith('exo_c_')]
past_exo_cat_cols = []
future_exo_cont_cols = [
    c for c in df.columns
    if c.startswith('exo_') and not c.startswith('exo_p_') and not c.startswith('exo_c_')
]

plan_week = resolve_single_plan_week(
    target_df,
    date_col=DATE_COL,
    horizon=horizon,
    plan_week=plan_week,
)

eval_ids = select_eval_ids_with_full_actual_coverage(
    target_df,
    id_col=ID_COL,
    date_col=DATE_COL,
    plan_weeks=[plan_week],
    horizon=horizon,
)

eval_target_df = target_df.filter(pl.col(ID_COL).cast(pl.String).is_in(eval_ids))
train_one_table = df.filter(pl.col(DATE_COL) < int(plan_week))
infer_one_table = df.filter(pl.col(ID_COL).cast(pl.String).is_in(eval_ids))

print('df shape             :', df.shape)
print('target_df shape      :', target_df.shape)
print('train_one_table      :', train_one_table.shape)
print('infer_one_table      :', infer_one_table.shape)
print('eval_target_df       :', eval_target_df.shape)
print('eval_id_count        :', len(eval_ids))
print('plan_week            :', plan_week)
print('past_exo_cont_cols   :', len(past_exo_cont_cols), past_exo_cont_cols)
print('past_exo_cat_cols    :', len(past_exo_cat_cols), past_exo_cat_cols)
print('future_exo_cont_cols :', len(future_exo_cont_cols), future_exo_cont_cols)
print('date min/max         :', int(df[DATE_COL].min()), int(df[DATE_COL].max()))



## Loader Smoke Test

먼저 세 모델 입력 계약이 Walmart 데이터에서도 정상인지 확인합니다.
- `patchtst_no_future`, `timexer`: past-only
- `exotst`: future exogenous 사용


In [ ]:

def build_walmart_data_request(
    *,
    df: pl.DataFrame,
    batch_size: int,
    shuffle: bool,
    use_future_exogenous: bool,
) -> DataRequest:
    return DataRequest(
        df=df,
        window=DataWindowConfig(
            lookback=lookback,
            horizon=horizon,
            freq=FREQ,
        ),
        columns=DataColumnConfig(
            id_col=ID_COL,
            date_col=DATE_COL,
            y_col=Y_COL,
        ),
        exogenous=ExogenousConfig(
            use_exogenous_mode=True,
            use_past_exogenous=True,
            use_future_exogenous=use_future_exogenous,
            past_exo_cont_cols=list(past_exo_cont_cols),
            past_exo_cat_cols=list(past_exo_cat_cols),
            future_exo_cont_cols=list(future_exo_cont_cols) if use_future_exogenous else [],
            future_exo_cb=None,
            part_future_exo_fn=None,
        ),
        loader=LoaderConfig(
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
            persistent_workers=persistent_workers,
            prefetch_factor=prefetch_factor,
        ),
    )


def inspect_model_batches(model_names: list[str]) -> None:
    for spec in resolve_walmart_model_specs(model_names):
        req = build_walmart_data_request(
            df=train_one_table,
            batch_size=train_batch_size,
            shuffle=True,
            use_future_exogenous=spec.use_future_exogenous,
        )
        dm = build_datamodule(req)
        loader = dm.get_train_loader()
        batch = next(iter(loader))
        x, y, uid, future_exo, past_exo_cont, past_exo_cat = batch
        print(f'[{spec.label}] x            :', tuple(x.shape))
        print(f'[{spec.label}] y            :', tuple(y.shape))
        print(f'[{spec.label}] future_exo   :', tuple(future_exo.shape))
        print(f'[{spec.label}] past_exo_cont:', tuple(past_exo_cont.shape))
        print(f'[{spec.label}] past_exo_cat :', tuple(past_exo_cat.shape))
        print('---')


inspect_model_batches(models_to_run)



## Train + Forecast

이 단계에서는 같은 global model family를 Walmart dense weekly 데이터에 올려서,
DSIO에서 보였던 강한 scale collapse / segment-specific pathology가 여기서도 반복되는지 확인합니다.


In [ ]:

def build_walmart_architecture(spec: ModelSpec):
    patchtst_mode = WALMART_PATCHTST_FUSION_MODES.get(spec.label)
    arch_args = argparse.Namespace(
        patch_len=13,
        stride=6,
        patchtst_d_model=64,
        patchtst_layers=3,
        patchtst_d_ff=128,
        patchtst_future_exo_fusion_mode=patchtst_mode or 'token_cross_attn',
        patchtst_future_exo_fusion_dropout=patchtst_future_exo_fusion_dropout,
        timexer_patch_len=13,
        timexer_d_model=128,
        timexer_heads=8,
        timexer_d_ff=256,
        timexer_e_layers=3,
        timexer_dropout=0.1,
        timexer_factor=5,
        timexer_activation='gelu',
        timexer_use_norm=True,
        exotst_d_model=128,
        exotst_heads=8,
        exotst_d_ff=256,
        exotst_dropout=0.1,
        exotst_attn_dropout=0.1,
        exotst_exo_enc_layers=2,
        exotst_fusion_layers=2,
        exotst_endo_dec_layers=2,
        exotst_exo_memory_mode='all',
        exotst_exo_nan_policy='zero+indicator',
        exotst_use_revin=True,
        exotst_subtract_last=True,
    )
    return build_architecture_config(arch_args)


run_root = ensure_dir(artifact_root / f'lb{lookback}_h{horizon}_pw{plan_week}')


def run_single_model(spec):
    model_dir = ensure_dir(run_root / spec.label)
    architecture = build_walmart_architecture(spec)

    train_req = build_walmart_data_request(
        df=train_one_table,
        batch_size=train_batch_size,
        shuffle=True,
        use_future_exogenous=spec.use_future_exogenous,
    )

    train_result = train(
        TrainRequest(
            data=train_req,
            freq=FREQ,
            use_exogenous_mode=True,
            use_past_exogenous=True,
            use_future_exogenous=spec.use_future_exogenous,
            models=[spec.request_key],
            architecture=architecture,
            trainer=TrainerConfig(
                warmup_epochs=warmup_epochs,
                spike_epochs=spike_epochs,
                lr=lr,
                use_intermittent=use_intermittent,
                val_use_weights=val_use_weights,
            ),
            ssl=SSLConfig(
                mode=effective_patchtst_ssl_mode if spec.request_key.startswith('patchtst') else 'off',
                pretrain_epochs=ssl_pretrain_epochs,
                mask_ratio=ssl_mask_ratio,
                loss_type=ssl_loss_type,
            ),
            runtime=RuntimeConfig(device=device),
            artifacts=ArtifactConfig(
                save_dir=str(model_dir),
                auto_save_dir=False,
            ),
        )
    )

    ckpt_path = train_result.primary_ckpt_path or train_result.ckpt_paths.get(spec.request_key)
    if not ckpt_path:
        raise RuntimeError(f'No checkpoint produced for {spec.label}')

    predictor = load_predictor(
        ckpt_path,
        device=device,
        forecaster_kwargs={
            'target_channel': 0,
            'fill_mode': 'copy_last',
            'use_winsor': False,
            'use_multi_guard': False,
        },
    )

    infer_req = build_walmart_data_request(
        df=infer_one_table,
        batch_size=infer_batch_size,
        shuffle=False,
        use_future_exogenous=spec.use_future_exogenous,
    )
    infer_dm = build_datamodule(infer_req)
    infer_loader = infer_dm.get_inference_loader_at_plan(int(plan_week))

    forecast_df = make_point_forecast_result_table(
        inference_loader=infer_loader,
        predictor=predictor,
        model_name=spec.label,
        plan_week=int(plan_week),
        horizon=horizon,
        device=device,
        max_parts=max_parts_per_plan,
    )
    forecast_df = attach_actuals(
        forecast_df,
        eval_target_df,
        id_col=ID_COL,
        date_col=DATE_COL,
        y_col=Y_COL,
    )
    return train_result, forecast_df


results = {}
forecast_tables = []

for spec in resolve_walmart_model_specs(models_to_run):
    print(f'=== RUN {spec.label} ({spec.request_key}) ===')
    print('patchtst_future_exo_fusion_mode:', WALMART_PATCHTST_FUSION_MODES.get(spec.label))
    train_result, forecast_df = run_single_model(spec)
    results[spec.label] = train_result
    forecast_tables.append(forecast_df)
    print(spec.label, 'forecast shape:', forecast_df.shape)

combined_forecast_df = pl.concat(forecast_tables, how='vertical_relaxed') if forecast_tables else pl.DataFrame()
combined_forecast_df.head()



## Metrics and Plots

여기서는 aggregate scale / part-level shape를 함께 봅니다.

해석 팁:
- `pred_to_actual_ratio ≈ 1`: total scale이 비교적 정상
- aggregate plot이 GT를 비슷하게 따라가면 global model family 자체는 크게 문제 없을 가능성이 큼
- 여기서도 DSIO처럼 전반적인 2x overforecast가 반복되면 모델/학습 정책을 더 의심해야 함


In [ ]:

def summarize_scale(df: pl.DataFrame, group_cols: list[str]) -> pl.DataFrame:
    valid = df.filter(pl.col('actual').is_not_null())
    if valid.height == 0:
        return pl.DataFrame()
    return (
        valid.group_by(group_cols)
        .agg([
            pl.len().alias('n_rows'),
            pl.col('prediction').mean().alias('pred_mean'),
            pl.col('prediction').sum().alias('pred_sum'),
            pl.col('actual').mean().alias('actual_mean'),
            pl.col('actual').sum().alias('actual_sum'),
        ])
        .with_columns(
            pl.when(pl.col('actual_sum').abs() > 1e-12)
            .then(pl.col('pred_sum') / pl.col('actual_sum'))
            .otherwise(None)
            .alias('pred_to_actual_ratio')
        )
        .sort(group_cols)
    )



overall_df, horizon_df, latest_summary_df = compute_metric_tables(combined_forecast_df)
latest_df = select_latest_revision(combined_forecast_df.filter(pl.col('actual').is_not_null()))
scale_df = summarize_scale(latest_df, ['model_name'])
sampled_parts = select_plot_parts(latest_df, plot_part_count=plot_part_count)

metric_plot_path = run_root / 'walmart_metric_summary.png'
agg_plot_path = run_root / 'walmart_latest_revision_aggregate.png'
part_plot_path = run_root / 'walmart_latest_revision_parts.png'

plot_metric_summary(overall_df, metric_plot_path)
plot_latest_revision_aggregate(latest_df, agg_plot_path)
plot_latest_revision_part_grid(
    latest_df,
    sampled_parts=sampled_parts,
    save_path=part_plot_path,
    ncols=3,
)

print('overall metrics')
display(overall_df)
print('latest revision metrics')
display(latest_summary_df)
print('latest revision scale summary')
display(scale_df)

for p in [metric_plot_path, agg_plot_path, part_plot_path]:
    if p.exists():
        display(Image(filename=str(p)))



## PatchTST Diagnostics

이 섹션은 Walmart sanity check에서 기본 비교 모델인 `patchtst_no_future`를 기준으로,
왜 PatchTST가 future exo를 붙였을 때 불안정해지는지 추가로 좁히기 위한 진단입니다.

확인하는 것:
- `patchtst_no_future` raw prediction에 음수/극단치가 있는지
- 동일한 아키텍처에서 `future exogenous`를 다시 켜면 `patchtst_with_future`가 얼마나 불안정해지는지

해석 포인트:
- `raw unclipped`에서 음수 비율이 높고 clipped plot에서 0이 자주 나오면,
  현재 문제는 PatchTST point head 출력의 불안정성일 가능성이 큽니다.
- `patchtst_with_future`가 `patchtst_no_future`보다 훨씬 나쁘면,
  Walmart에서는 future exo handling이 핵심 원인이라는 강한 신호로 볼 수 있습니다.


In [ ]:

patch_compare_models = [
    'patchtst_no_future',
    'patchtst_head_flatten',
    'patchtst_token_cross_attn',
]
patch_compare_df = combined_forecast_df.filter(pl.col('model_name').is_in(patch_compare_models))
patch_compare_latest_df = select_latest_revision(patch_compare_df.filter(pl.col('actual').is_not_null()))
patch_compare_metrics = compute_metric_tables(patch_compare_df)[2].sort('wape')
patch_compare_scale = summarize_scale(patch_compare_latest_df, ['model_name'])

print('PatchTST future-exo fusion comparison metrics')
display(patch_compare_metrics)
print('PatchTST future-exo fusion comparison scale')
display(patch_compare_scale)

patch_metric_plot_path = run_root / 'walmart_patchtst_future_fusion_metric_summary.png'
patch_agg_plot_path = run_root / 'walmart_patchtst_future_fusion_aggregate.png'
patch_part_plot_path = run_root / 'walmart_patchtst_future_fusion_parts.png'

plot_metric_summary(patch_compare_metrics, patch_metric_plot_path)
plot_latest_revision_aggregate(patch_compare_latest_df, patch_agg_plot_path)
patch_sampled_parts = select_plot_parts(patch_compare_latest_df, plot_part_count=plot_part_count)
plot_latest_revision_part_grid(
    patch_compare_latest_df,
    sampled_parts=patch_sampled_parts,
    save_path=patch_part_plot_path,
    ncols=3,
)

for p in [patch_metric_plot_path, patch_agg_plot_path, patch_part_plot_path]:
    if p.exists():
        display(Image(filename=str(p)))



## Interpretation Guide

이 notebook의 목적은 **DSIO 문제의 원인이 모델 family 자체인지**를 분리하는 것입니다.

대략 이렇게 해석하면 됩니다.

- Walmart에서 세 모델 모두 `pred_to_actual_ratio`가 1 근처고 aggregate / part plot도 비교적 정상이다
  - global model family 자체는 작동한다고 볼 수 있음
  - DSIO 문제는 `small / erratic / high-obsolescence / decline` 같은 세그먼트 특성 쪽 가능성이 큼

- Walmart에서도 세 모델이 비슷하게 강한 overforecast / underforecast를 만든다
  - 모델/학습 정책(lookback, loss, stage schedule, calibration) 쪽을 더 강하게 의심해야 함

- Walmart에서는 한 모델만 정상이고 나머지 둘은 무너진다
  - 특정 architecture family가 현재 train policy와 더 잘/덜 맞는 것일 수 있음
